# Phase 3: Map Gene sang Protein

Notebook này đọc kết quả DEG từ Phase 2, chỉ giữ các gene DEG, chuẩn hóa tên gene và map sang protein bằng STRING gene_map. Output sẽ dùng cho các bước protein-protein interaction phía sau.

Nguyên tắc chạy:
- Chỉ đọc `gdc_deg_result` từ analysis và `STRING/gene_map` từ refined.
- Không sửa raw/refined.
- Ưu tiên đọc Hive table `STRING.gene_map` nếu đã đăng ký; nếu chưa có thì fallback sang Parquet refined.
- Ghi output vào `hdfs://master11:9000/drugtarget/data/analysis/deg_mapped_proteins`.

In [1]:
from pyspark.sql import SparkSession, functions as F

# Khai báo đường dẫn HDFS và tên bảng phục vụ Phase 3.
HDFS_BASE = "hdfs://master11:9000"
DEG_INPUT_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/gdc_deg_result"
GENE_MAP_TABLE = "STRING.gene_map"
GENE_MAP_PATH = f"{HDFS_BASE}/drugtarget/data/refined/STRING/gene_map"
MAPPED_OUTPUT_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/deg_mapped_proteins"

# Tạo SparkSession để map gene DEG sang STRING protein.
spark = (
    SparkSession.builder.appName("drugtarget-gdc-phase3-map-gene-to-protein")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .enableHiveSupport()
    .getOrCreate()
)

# Giảm log để notebook dễ đọc hơn.
spark.sparkContext.setLogLevel("WARN")

print(f"Spark master: {spark.sparkContext.master}")
print(f"Input DEG Phase 3: {DEG_INPUT_PATH}")
print(f"Input STRING gene_map fallback: {GENE_MAP_PATH}")
print(f"Output Phase 3: {MAPPED_OUTPUT_PATH}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/03 08:14:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark master: local[*]
Input DEG Phase 3: hdfs://master11:9000/drugtarget/data/analysis/gdc_deg_result
Input STRING gene_map fallback: hdfs://master11:9000/drugtarget/data/refined/STRING/gene_map
Output Phase 3: hdfs://master11:9000/drugtarget/data/analysis/deg_mapped_proteins


In [2]:
def read_table_or_parquet(table_name: str, parquet_path: str):
    """Đọc Hive table nếu có; nếu chưa có thì đọc Parquet refined trên HDFS."""
    try:
        frame = spark.table(table_name)
        print(f"Đọc Hive table thành công: {table_name}")
        return frame
    except Exception as exc:
        print(f"Không đọc được Hive table {table_name}; chuyển sang Parquet refined.")
        print(f"Lý do: {type(exc).__name__}: {exc}")
        print(f"Đọc Parquet: {parquet_path}")
        return spark.read.parquet(parquet_path)


def require_columns(frame, required_columns, frame_name: str) -> None:
    """Dừng pipeline nếu DataFrame thiếu cột bắt buộc."""
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    if missing_columns:
        missing_text = ", ".join(missing_columns)
        raise ValueError(f"{frame_name} thiếu cột bắt buộc: {missing_text}")


# Đọc DEG Phase 2 và bảng map gene-protein của STRING.
deg = spark.read.parquet(DEG_INPUT_PATH)
gene_map = read_table_or_parquet(GENE_MAP_TABLE, GENE_MAP_PATH)

DEG_REQUIRED_COLUMNS = [
    "gene_id_base",
    "gene_name",
    "log2FC",
    "p_value",
    "deg_direction",
    "is_deg",
]
GENE_MAP_REQUIRED_COLUMNS = ["protein_id", "ensp_id", "gene_confidence"]
require_columns(deg, DEG_REQUIRED_COLUMNS, "gdc_deg_result")
require_columns(gene_map, GENE_MAP_REQUIRED_COLUMNS, "STRING.gene_map")

if "gene_name_norm" not in gene_map.columns and "gene_name" not in gene_map.columns:
    raise ValueError("STRING.gene_map cần có gene_name_norm hoặc gene_name để join.")

print("Schema DEG Phase 2:")
deg.printSchema()
print("Schema STRING gene_map:")
gene_map.printSchema()

26/06/03 08:14:37 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/03 08:14:37 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


26/06/03 08:14:40 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/06/03 08:14:40 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dis@100.82.104.4


26/06/03 08:14:41 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


26/06/03 08:14:41 WARN ObjectStore: Failed to get database string, returning NoSuchObjectException


Không đọc được Hive table STRING.gene_map; chuyển sang Parquet refined.
Lý do: AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `STRING`.`gene_map` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [STRING, gene_map], [], false

Đọc Parquet: hdfs://master11:9000/drugtarget/data/refined/STRING/gene_map


Schema DEG Phase 2:
root
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- mean_tumor: double (nullable = true)
 |-- mean_normal: double (nullable = true)
 |-- mean_log2_tpm_tumor: double (nullable = true)
 |-- mean_log2_tpm_normal: double (nullable = true)
 |-- log2FC: double (nullable = true)
 |-- p_value: double (nullable = true)
 |-- deg_direction: string (nullable = true)
 |-- is_deg: boolean (nullable = true)

Schema STRING gene_map:
root
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- gene_id: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_name_norm: string (nullable = true)
 |-- gene_confidence: string (nullable = true)



In [3]:
# Chuẩn hóa DEG gene_name và chỉ giữ các gene được đánh dấu là DEG.
deg_clean = (
    deg.filter(F.col("is_deg") == F.lit(True))
    .filter(F.col("gene_name").isNotNull())
    .withColumn("gene_name_norm", F.upper(F.trim(F.col("gene_name"))))
    .filter(F.col("gene_name_norm").isNotNull() & (F.col("gene_name_norm") != F.lit("")))
    .select("gene_id_base", "gene_name", "gene_name_norm", "log2FC", "p_value", "deg_direction")
    .distinct()
)

# Ưu tiên gene_name_norm của STRING, fallback sang gene_name nếu gene_name_norm rỗng hoặc null.
gene_map_name_exprs = []
if "gene_name_norm" in gene_map.columns:
    gene_map_name_exprs.append(F.when(F.trim(F.col("gene_name_norm")) != F.lit(""), F.col("gene_name_norm")))
if "gene_name" in gene_map.columns:
    gene_map_name_exprs.append(F.when(F.trim(F.col("gene_name")) != F.lit(""), F.col("gene_name")))

map_join_name = F.upper(F.trim(F.coalesce(*gene_map_name_exprs)))

gene_map_clean = (
    gene_map.withColumn("gene_name_norm_join", map_join_name)
    .filter(F.col("protein_id").isNotNull())
    .filter(F.col("gene_name_norm_join").isNotNull() & (F.col("gene_name_norm_join") != F.lit("")))
    .select("protein_id", "ensp_id", "gene_confidence", "gene_name_norm_join")
    .distinct()
)

num_deg = deg_clean.select("gene_name_norm").distinct().count()
if num_deg == 0:
    raise ValueError("Không có DEG nào để map sang protein.")

# Join DEG với STRING gene_map bằng tên gene đã chuẩn hóa.
OUTPUT_COLUMNS = [
    "gene_id_base",
    "gene_name",
    "protein_id",
    "ensp_id",
    "gene_confidence",
    "log2FC",
    "p_value",
    "deg_direction",
]

deg_mapped_proteins = (
    deg_clean.alias("d")
    .join(
        gene_map_clean.alias("m"),
        F.col("d.gene_name_norm") == F.col("m.gene_name_norm_join"),
        "inner",
    )
    .select(
        F.col("d.gene_id_base"),
        F.col("d.gene_name"),
        F.col("m.protein_id"),
        F.col("m.ensp_id"),
        F.col("m.gene_confidence"),
        F.col("d.log2FC"),
        F.col("d.p_value"),
        F.col("d.deg_direction"),
    )
    .distinct()
    .cache()
)

mapped_row_count = deg_mapped_proteins.count()
if mapped_row_count == 0:
    raise ValueError("Không map được DEG nào sang STRING protein.")

print("Schema output Phase 3:")
deg_mapped_proteins.printSchema()
print(f"Số dòng mapping gene-protein: {mapped_row_count}")

Schema output Phase 3:
root
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- gene_confidence: string (nullable = true)
 |-- log2FC: double (nullable = true)
 |-- p_value: double (nullable = true)
 |-- deg_direction: string (nullable = true)

Số dòng mapping gene-protein: 2579


In [4]:
# Ghi kết quả map protein vào layer analysis, không ghi vào raw/refined.
deg_mapped_proteins.write.mode("overwrite").option("compression", "snappy").parquet(MAPPED_OUTPUT_PATH)
print(f"Đã ghi output Phase 3: {MAPPED_OUTPUT_PATH}")

Đã ghi output Phase 3: hdfs://master11:9000/drugtarget/data/analysis/deg_mapped_proteins


In [5]:
# Kiểm tra output path tồn tại trên HDFS sau khi ghi.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
output_hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(MAPPED_OUTPUT_PATH)
output_fs = output_hdfs_path.getFileSystem(hadoop_conf)
if not output_fs.exists(output_hdfs_path):
    raise AssertionError(f"Output Phase 3 chưa tồn tại trên HDFS: {MAPPED_OUTPUT_PATH}")

# Đọc lại output để xác nhận schema và chất lượng mapping.
mapped_output = spark.read.parquet(MAPPED_OUTPUT_PATH)
if mapped_output.columns != OUTPUT_COLUMNS:
    raise AssertionError(f"Schema Phase 3 không đúng: {mapped_output.columns}")

null_protein_count = mapped_output.filter(F.col("protein_id").isNull()).count()
if null_protein_count != 0:
    raise AssertionError(f"Có {null_protein_count} dòng protein_id null trong output Phase 3.")

num_mapped = mapped_output.select(F.upper(F.trim(F.col("gene_name"))).alias("gene_name_norm")).distinct().count()
mapping_rate = num_mapped / num_deg if num_deg else 0.0

print("Kiểm tra output Phase 3 hoàn tất.")
print(f"DEG genes: {num_deg}")
print(f"Mapped genes: {num_mapped}")
print(f"Mapping rate: {mapping_rate:.4f}")
print("Số protein map được theo hướng DEG:")
mapped_output.groupBy("deg_direction").agg(
    F.count("*").alias("num_rows"),
    F.countDistinct("gene_name").alias("num_genes"),
    F.countDistinct("protein_id").alias("num_proteins"),
).orderBy("deg_direction").show(truncate=False)

print("Một số mapping đầu tiên:")
mapped_output.orderBy(F.desc(F.abs(F.col("log2FC")))).show(20, truncate=False)

Kiểm tra output Phase 3 hoàn tất.
DEG genes: 2598
Mapped genes: 2579
Mapping rate: 0.9927
Số protein map được theo hướng DEG:


+-------------+--------+---------+------------+
|deg_direction|num_rows|num_genes|num_proteins|
+-------------+--------+---------+------------+
|Downregulated|1677    |1677     |1677        |
|Upregulated  |902     |902      |902         |
+-------------+--------+---------+------------+

Một số mapping đầu tiên:


+---------------+---------+--------------------+---------------+---------------+-------------------+-----------------------+-------------+
|gene_id_base   |gene_name|protein_id          |ensp_id        |gene_confidence|log2FC             |p_value                |deg_direction|
+---------------+---------+--------------------+---------------+---------------+-------------------+-----------------------+-------------+
|ENSG00000168484|SFTPC    |9606.ENSP00000316152|ENSP00000316152|high           |-8.190919857851668 |0.0                    |Downregulated|
|ENSG00000204305|AGER     |9606.ENSP00000364210|ENSP00000364210|high           |-6.566993363624583 |0.0                    |Downregulated|
|ENSG00000066405|CLDN18   |9606.ENSP00000340939|ENSP00000340939|high           |-6.324261953457807 |0.0                    |Downregulated|
|ENSG00000170323|FABP4    |9606.ENSP00000256104|ENSP00000256104|high           |-5.667173870209467 |0.0                    |Downregulated|
|ENSG00000149021|SCGB1A1  |

In [6]:
# Giải phóng cache sau khi hoàn tất Phase 3.
deg_mapped_proteins.unpersist()
print("Hoàn tất Phase 3: Map Gene sang Protein")

Hoàn tất Phase 3: Map Gene sang Protein
